**Chuẩn bị dữ liệu và cài đặt**

In [ ]:
pip install pandasql

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=b933bd2936e7ba22c3b628da6282231f0a1ef4ad48a11c7503ff9fa07e3230b6
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


**1. Kết nối hai bảng**

In [ ]:
import pandas as pd
from pandasql import sqldf
import datetime

# Thiết lập hiển thị của Pandas để bảng trông đẹp hơn
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Khởi tạo dữ liệu
student = pd.DataFrame({
    'student_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'name': ['Nguyen Minh Hoang', 'Tran Thi Lan', 'Pham Van Nam', 'Le Thanh Huyen', 'Vu Quoc Anh',
             'Dang Thuy Linh', 'Bui Tien Dung', 'Ho Ngoc Mai', 'Duong Huu Phuc', 'Cao Thi Hanh'],
    'class': ['May Tinh', 'Kinh Te', 'Toan Tin', 'Toan Tin', 'May Tinh',
              'May Tinh', 'Kinh Te', 'Toan Tin', 'Kinh Te', 'May Tinh'],
    'course_id': [12, 34, None, 20, 24, 24, 34, 20, None, None],
    'score': [6.7, 9.2, 7.9, 7.2, 8.0, 5.5, 9.2, 8.8, 7.2, 7.0]
})

course = pd.DataFrame({
    'id': [12, 34, 26],
    'course_name': ['Giai tich', 'Thong ke', 'Tin hoc']
})

pysqldf = lambda q: sqldf(q, globals())

print("--- BẢNG STUDENT GỐC ---")
display(student)
print("\n--- BẢNG COURSE GỐC ---")
display(course)

--- BẢNG STUDENT GỐC ---


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0



--- BẢNG COURSE GỐC ---


,id,course_name
0,12,Giai tich
1,34,Thong ke
2,26,Tin hoc


**b .Kết nối bảng (Join)**

In [ ]:
query_cross = "SELECT * FROM student CROSS JOIN course"
df_cross = pysqldf(query_cross)

In [ ]:
from IPython.display import display
#  TÍCH DESCARTES (CROSS JOIN)
# Mỗi dòng của bảng Student kết hợp với mọi dòng của bảng Course (10 SV x 3 Môn = 30 dòng)
print(" Kết quả tích DESCARTES (Hiển thị 5 dòng đầu) ---")
df_cross = pysqldf("SELECT * FROM student CROSS JOIN course")
display(df_cross.head(5))

 Kết quả tích DESCARTES (Hiển thị 5 dòng đầu) ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke


 **Tích Descartes (CROSS JOIN)**
 Kỹ thuật: Phép toán tổ hợp mọi dòng của bảng A với mọi dòng của bảng B mà không cần điều kiện kết nối.Dữ liệu thực tế: Tạo ra $10 \times 3 = 30$ bản ghi.Nhận xét: Đây là phép Join có độ nhiễu cao nhất. Nó tạo ra những bản ghi vô lý (ví dụ: Sinh viên chuyên ngành "Kinh tế" nhưng lại bị ghép cặp với môn "Giải tích" của khoa Toán).

In [ ]:
from IPython.display import display

# 1. INNER JOIN
# Chỉ lấy những bản ghi có course_id khớp nhau ở cả hai bảng
print("KẾT QUẢ INNER JOIN ")
df_inner = pysqldf("SELECT s.student_id, s.name, s.class, s.course_id, c.course_name FROM student s INNER JOIN course c ON s.course_id = c.id")
display(df_inner)


KẾT QUẢ INNER JOIN 


,student_id,name,class,course_id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,Thong ke
2,7,Bui Tien Dung,Kinh Te,34.0,Thong ke


**INNER JOIN (Kết nối nội)**
Kỹ thuật: Chỉ giữ lại các dòng mà giá trị course_id ở bảng Student xuất hiện chính xác trong bảng Course.

Dữ liệu thực tế: Chỉ còn 3 dòng (Hoàng, Lan, Dũng).

Nhận xét: * Phép Join này cho thấy sự thiếu hụt dữ liệu nghiêm trọng trong hệ thống. Có đến 70% sinh viên bị "biến mất" khỏi kết quả vì mã môn học của họ (20, 24) hoặc trạng thái chưa đăng ký (None) không tồn tại trong danh mục môn học.

Nó bộc lộ lỗi logic: Sinh viên đang học những mã môn (20, 24) chưa được định nghĩa chính thức.

Kết luận: Dùng để lọc ra danh sách sinh viên "hợp lệ" về mặt dữ liệu.

In [ ]:
# 2. LEFT JOIN
# Lấy tất cả sinh viên, môn nào không có trong bảng Course sẽ hiện NULL (None)
print(" KẾT QUẢ LEFT JOIN")
df_left = pysqldf("SELECT s.student_id, s.name, s.class, s.course_id, c.course_name FROM student s LEFT JOIN course c ON s.course_id = c.id")
display(df_left)


 KẾT QUẢ LEFT JOIN


,student_id,name,class,course_id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,None
4,5,Vu Quoc Anh,May Tinh,24.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,None
6,7,Bui Tien Dung,Kinh Te,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,None
8,9,Duong Huu Phuc,Kinh Te,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,None


**LEFT JOIN (Kết nối trái)**
Kỹ thuật: Ưu tiên giữ toàn bộ bảng bên trái (Student), bảng bên phải (Course) chỉ bổ sung thông tin nếu khớp mã.

Dữ liệu thực tế: Giữ đủ 10 dòng. Các cột từ bảng Course sẽ bị NaN (NULL) ở những dòng không khớp.

Nhận xét: * Đây là phép Join trung thực nhất với thực trạng dữ liệu. Nó cho thấy sinh viên nào đã có tên môn học, sinh viên nào đang học mã môn "lạ", và sinh viên nào chưa có mã môn.

Nó không làm mất dữ liệu của đối tượng chính (Sinh viên).

Kết luận: Là công cụ tốt nhất để kiểm toán dữ liệu (Data Auditing) và lập danh sách tổng hợp.

In [ ]:
# 3. RIGHT JOIN ---
# Lấy tất cả các môn học, môn nào không có sinh viên học sẽ hiện NULL ở phần sinh viên
print("KẾT QUẢ RIGHT JOIN")
# Vì SQLite/pandasql không hỗ trợ RIGHT JOIN trực tiếp, ta đảo bảng Course lên trước
df_right = pysqldf("SELECT c.id as course_id, c.course_name, s.name, s.class FROM course c LEFT JOIN student s ON c.id = s.course_id")
display(df_right)


KẾT QUẢ RIGHT JOIN


,course_id,course_name,name,class
0,12,Giai tich,Nguyen Minh Hoang,May Tinh
1,34,Thong ke,Bui Tien Dung,Kinh Te
2,34,Thong ke,Tran Thi Lan,Kinh Te
3,26,Tin hoc,None,None


**RIGHT JOIN (Kết nối phải)**
Kỹ thuật: Ưu tiên giữ toàn bộ bảng bên phải (Course), bảng bên trái (Student) chỉ xuất hiện nếu có người học.

Dữ liệu thực tế: Xuất hiện môn "Tin học" (ID 26) kèm theo các cột thông tin sinh viên là NULL.

Nhận xét: * Tiết lộ thông tin về môn học không có người học. Trong khi sinh viên đang thiếu môn, thì môn "Tin học" lại không được ai đăng ký.

Ngược lại, nó sẽ làm mất thông tin của những sinh viên học mã môn 20, 24 vì những mã này không nằm trong bảng Course.

Kết luận: Dùng để phân tích hiệu quả mở lớp hoặc danh mục hàng hóa/dịch vụ tồn kho.

In [ ]:
# 4. FULL OUTER JOIN
# Lấy tất cả bản ghi từ cả hai bảng (Dùng hàm merge của Pandas)
print(" KẾT QUẢ FULL OUTER JOIN")
df_full = pd.merge(student, course, left_on='course_id', right_on='id', how='outer')
display(df_full)

 KẾT QUẢ FULL OUTER JOIN


,student_id,name,class,course_id,score,id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,NaN
2,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,NaN
3,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,NaN
4,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,26.0,Tin hoc
6,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
8,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,NaN
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,NaN


**FULL OUTER JOIN (Kết nối toàn phần)**
Kỹ thuật: Kết hợp cả Left và Right Join, không bỏ sót bất kỳ dòng nào từ cả hai phía.

Dữ liệu thực tế: Hiển thị đủ 10 sinh viên + môn học "Tin học" không có người học.

Nhận xét: * Đây là bức tranh đầy đủ nhất về mối quan hệ giữa hai bảng.

Nó vạch trần lỗ hổng ở cả hai phía: Phía Student có những bạn học mã môn sai, phía Course có những môn chưa được khai thác.

Kết luận: Dùng khi cần đối soát tổng thể hệ thống để làm sạch dữ liệu (Data Cleaning).

**2. Cập nhật dữ liệu và Thống kê**

In [ ]:
# Xóa các course_id không tồn tại
query = """
DELETE FROM student
WHERE course_id IS NOT NULL
AND course_id NOT IN (SELECT id FROM course)
"""
conn.execute(query)

# Cập nhật course_id NULL -> gán mặc định = 12
conn.execute("UPDATE student SET course_id = 12 WHERE course_id IS NULL")

**Thống kê Theo lớp**

In [ ]:
query = """
SELECT class, COUNT(*) AS total_students, AVG(score) AS avg_score
FROM student
GROUP BY class
"""
pd.read_sql(query, conn)

,class,total_students,avg_score
0,Kinh Tế,3,8.533333
1,Máy Tính,2,6.850000
2,Toán Tin,1,7.900000


Kỹ thuật: Kết hợp tốt giữa SQL (GROUP BY, AVG) và Pandas để tổng hợp dữ liệu từ bảng thô sang bảng báo cáo.

Kết quả phân tích: * Lớp Kinh tế: Dẫn đầu cả về số lượng (3 SV) và chất lượng (Điểm TB cao nhất: 8.53).

Lớp Máy tính: Có điểm trung bình thấp nhất (6.85), cần chú ý cải thiện kết quả học tập.

In [ ]:
# Theo môn học
query = """
SELECT c.course_name, COUNT(*) AS total_students, AVG(s.score) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
pd.read_sql(query, conn)

,course_name,total_students,avg_score
0,Giai tich,4,7.2
1,Thong ke,2,9.2


Thống kê theo Môn học:

Môn Thống kê: Mặc dù chỉ có 2 sinh viên theo học nhưng đạt điểm trung bình tuyệt đối là 9.2. Điều này cho thấy chất lượng tiếp thu của sinh viên ở môn học này rất cao.

Môn Giải tích: Có số lượng sinh viên đăng ký đông hơn (4 bạn) nhưng điểm trung bình ở mức 7.2, cho thấy mức độ khó hoặc khả năng tiếp cận của sinh viên với môn học này ở mức trung bình khá.

Đánh giá kỹ thuật: Việc sử dụng thành công phép JOIN giữa hai bảng student và course đã giúp kết nối mã môn học với tên môn học thực tế, biến dữ liệu thô thành thông tin có ý nghĩa quản lý

**Phân loại thi đua**

In [ ]:
print("\n=== PHAN LOAI THI DUA ===")
df_rank_course = pd.read_sql("""
SELECT c.course_name,
ROUND(AVG(s.score),2) AS avg_score,
CASE
    WHEN AVG(s.score) > 9 THEN 'Xuat sac'
    WHEN AVG(s.score) >= 6 THEN 'Tot'
    ELSE 'Kem'
END AS xep_loai
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)
display(df_rank_course)


=== PHAN LOAI THI DUA ===


,course_name,avg_score,xep_loai
0,Giai tich,7.2,Tot
1,Thong ke,9.2,Xuat sac


Câu lệnh đã thực hiện thành công việc kết nối (JOIN) hai bảng dữ liệu để lấy được tên môn học thay vì chỉ hiển thị mã số khô khan.

Việc sử dụng hàm ROUND(AVG(s.score), 2) giúp dữ liệu điểm số trở nên gọn gàng, chuyên nghiệp và dễ đọc.

Cấu trúc CASE WHEN đã thực hiện chính xác chức năng Phân loại (Classification), giúp chuyển đổi từ điểm số định lượng sang danh hiệu định tính một cách tự động.

Về mặt Nội dung/Insight:

Môn Thống kê: Đạt thành tích xuất sắc với điểm trung bình ấn tượng là 9.2. Điều này phản ánh năng lực học tập rất cao của nhóm sinh viên đăng ký môn học này.

Môn Giải tích: Đạt mức điểm trung bình là 7.2, hệ thống tự động xếp loại "Tot". Kết quả này nằm trong ngưỡng an toàn nhưng cho thấy sự phân hóa rõ rệt về độ khó hoặc mức độ tiếp thu so với môn Thống kê.

**3. Xếp hạng sinh viên (Top 3 cao nhất/thấp nhất)**

**Toàn bộ**

In [ ]:
print(" XẾP HẠNG TOÀN BỘ")
df_rank = pd.read_sql("""
SELECT *,
RANK() OVER (ORDER BY score DESC) AS rank_all
FROM student
""", conn)
display(df_rank)

print("Top 3 cao nhất:")
display(df_rank.sort_values("score", ascending=False).head(3))

print("Top 3 thấp nhất:")
display(df_rank.sort_values("score").head(3))

 XẾP HẠNG TOÀN BỘ


,student_id,name,class,course_id,score,rank_all
0,2,Trần Thị Lan,Kinh Tế,34.0,9.2,1
1,7,Bùi Tiến Dũng,Kinh Tế,34.0,9.2,1
2,3,Phạm Văn Nam,Toán Tin,12.0,7.9,3
3,9,Dương Hữu Phúc,Kinh Tế,12.0,7.2,4
4,10,Cao Thị Hạnh,Máy Tính,12.0,7.0,5
5,1,Nguyễn Minh Hoàng,Máy Tính,12.0,6.7,6


Top 3 cao nhất:


,student_id,name,class,course_id,score,rank_all
0,2,Trần Thị Lan,Kinh Tế,34.0,9.2,1
1,7,Bùi Tiến Dũng,Kinh Tế,34.0,9.2,1
2,3,Phạm Văn Nam,Toán Tin,12.0,7.9,3


Top 3 thấp nhất:


,student_id,name,class,course_id,score,rank_all
5,1,Nguyễn Minh Hoàng,Máy Tính,12.0,6.7,6
4,10,Cao Thị Hạnh,Máy Tính,12.0,7.0,5
3,9,Dương Hữu Phúc,Kinh Tế,12.0,7.2,4


**Theo lớp**

In [ ]:
print("Xếp hạng theo lớp")
df_rank_class = pd.read_sql("""
SELECT *,
RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_class
FROM student
""", conn)
display(df_rank_class)

Xếp hạng theo lớp


,student_id,name,class,course_id,score,rank_class
0,2,Trần Thị Lan,Kinh Tế,34.0,9.2,1
1,7,Bùi Tiến Dũng,Kinh Tế,34.0,9.2,1
2,9,Dương Hữu Phúc,Kinh Tế,12.0,7.2,3
3,10,Cao Thị Hạnh,Máy Tính,12.0,7.0,1
4,1,Nguyễn Minh Hoàng,Máy Tính,12.0,6.7,2
5,3,Phạm Văn Nam,Toán Tin,12.0,7.9,1


**Theo môn**

In [ ]:
print("Xếp hạng theo môn")
df_rank_course = pd.read_sql("""
SELECT s.*, c.course_name,
RANK() OVER (PARTITION BY c.course_name ORDER BY score DESC) AS rank_course
FROM student s
JOIN course c ON s.course_id = c.id
""", conn)
display(df_rank_course)

Xếp hạng theo môn


,student_id,name,class,course_id,score,course_name,rank_course
0,3,Phạm Văn Nam,Toán Tin,12.0,7.9,Giai tich,1
1,9,Dương Hữu Phúc,Kinh Tế,12.0,7.2,Giai tich,2
2,10,Cao Thị Hạnh,Máy Tính,12.0,7.0,Giai tich,3
3,1,Nguyễn Minh Hoàng,Máy Tính,12.0,6.7,Giai tich,4
4,2,Trần Thị Lan,Kinh Tế,34.0,9.2,Thong ke,1
5,7,Bùi Tiến Dũng,Kinh Tế,34.0,9.2,Thong ke,1


**4. Bổ sung trường Graduation Date**

In [43]:
print(" THÊM GRADUATION DATE ")
df_final = pd.read_sql("SELECT * FROM student", conn)
# Xếp hạng theo điểm
df_final["rank"] = df_final["score"].rank(ascending=False)

# Tính ngày tốt nghiệp
df_final["graduation_date"] = df_final["rank"].apply(
    lambda x: (datetime.datetime.now() + datetime.timedelta(days=int(x))).strftime("%Y-%m-%d")
)
# Lưu lại
df_final.to_sql("student", conn, index=False, if_exists="replace")
display(df_final)

 THÊM GRADUATION DATE 


,student_id,name,class,course_id,score,rank,graduation_date
0,1,Nguyễn Minh Hoàng,Máy Tính,12.0,6.7,6.0,2026-04-09
1,2,Trần Thị Lan,Kinh Tế,34.0,9.2,1.5,2026-04-04
2,3,Phạm Văn Nam,Toán Tin,12.0,7.9,3.0,2026-04-06
3,7,Bùi Tiến Dũng,Kinh Tế,34.0,9.2,1.5,2026-04-04
4,9,Dương Hữu Phúc,Kinh Tế,12.0,7.2,4.0,2026-04-07
5,10,Cao Thị Hạnh,Máy Tính,12.0,7.0,5.0,2026-04-08


**Nhận xét:**
Về mặt Kỹ thuật:

Xếp hạng (Ranking): Việc sử dụng hàm .rank(ascending=False) trên cột score đã phản ánh chính xác thứ bậc của sinh viên. Điểm cao nhất (9.2) tương ứng với hạng thấp nhất (1.5 - do có hai bạn đồng hạng), giúp việc định danh "Top" sinh viên trở nên dễ dàng.

Xử lý thời gian (Datetime): Mã nguồn đã kết hợp nhuần nhuyễn thư viện datetime và timedelta. Việc sử dụng hàm lambda để tính toán ngày tốt nghiệp dựa trên biến rank là một cách tiếp cận thông minh, biến giá trị xếp hạng thành một tham số có ý nghĩa thực tế.

Đồng bộ hóa: Lệnh .to_sql(..., if_exists="replace") đảm bảo cơ sở dữ liệu luôn được cập nhật phiên bản mới nhất, sẵn sàng cho các báo cáo định kỳ.

Về mặt Nội dung/Insight:

Tính công bằng: Kết quả cho thấy một logic quản lý thú vị: sinh viên có thành tích học tập càng tốt (hạng nhỏ) thì ngày tốt nghiệp dự kiến càng sớm. Ví dụ: Bạn Trần Thị Lan và Bùi Tiến Dũng (hạng 1.5) dự kiến tốt nghiệp vào ngày 04/04/2026, trong khi bạn Nguyễn Minh Hoàng (hạng 6) phải đến ngày 09/04/2026.

Dữ liệu thực tế: Bảng kết quả cuối cùng đã hiển thị đầy đủ các cột thông tin mới, giúp bức tranh về quản lý sinh viên trở nên đa chiều: từ điểm số, thứ hạng đến mốc thời gian quan trọng.